# TitanAgent 実験1.5: 小型 Titans (MAC) の段階的学習（SFT → GRPO）

**Kaggle 無料枠（T4 16GB / P100 16GB）で OOM を防ぎながら完走できる**、3段階カリキュラム（検索 → 概念の連鎖整理 → 記憶の再利用）の実験ノートブックです。

### 学習する3段階カリキュラム（ミニ勉強ループ）
1. **Level 1（単発検索＋整理）**: 不明な概念を1つ特定 → `<search>` で調べる → 整理して `<answer>`
2. **Level 2（概念の連鎖 / 2段階論理整理）**: Aを調べたら前提概念Bが不明だと気づく → Bも調べて論理的につなげる
3. **Level 3（記憶の再利用）**: 前半で調べて覚えた概念を、後半の別質問では**検索し直さず**記憶から答える

### 2段階トレーニング
- **Stage 1（SFT / 基礎作り）**: お手本データで `<think>` / `<search>` / `<answer>` の型と基礎推論を習得（プロンプトと `<result>` は損失から除外）
- **Stage 2（GRPO / 誤り前提の強化）**: 疑似検索環境と実際にやり取りして複数ロールアウトを生成し、形式・正解・記憶再利用の自動報酬で強化（1ロールアウトずつ勾配蓄積して VRAM を最小化）

In [ ]:
# 1. 環境確認 & ライブラリインストール
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip -q install -U titans-pytorch

In [ ]:
# 2. 実験設定（Kaggle T4 16GB で OOM しない安全デフォルト）
from pathlib import Path

PRESET = 'tiny'          # 'micro' (超高速確認), 'tiny' (推奨・~6M), 'small' (~16M)
SFT_STEPS = 800          # Stage 1 (SFT): まずは 600〜1000 step が目安
GRPO_STEPS = 120         # Stage 2 (GRPO): 100〜200 step が目安
GROUP_SIZE = 4           # 1問あたりの生成本数 (4 が 16GB VRAM で安全)
MAX_NEW_TOKENS = 160     # RL生成の最大長
KL_BETA = 0.04           # SFT基礎が崩れないためのKL制約
EVAL_PER_LEVEL = 15      # 各レベルの評価問題数
SEED = 42

if IN_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/checkpoints/exp1_curriculum_rl')
else:
    OUTPUT_DIR = Path('./checkpoints/exp1_curriculum_rl')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    'micro': dict(dim=128, depth=3, segment_len=32, num_persist_mem_tokens=4, num_longterm_mem_tokens=8,  seq_len=320, sft_batch_size=8, sft_grad_accum=2,  sft_lr=1.5e-3, grpo_lr=1e-4),
    'tiny':  dict(dim=256, depth=4, segment_len=64, num_persist_mem_tokens=4, num_longterm_mem_tokens=16, seq_len=384, sft_batch_size=4, sft_grad_accum=8,  sft_lr=8e-4,   grpo_lr=5e-5),
    'small': dict(dim=384, depth=6, segment_len=64, num_persist_mem_tokens=8, num_longterm_mem_tokens=16, seq_len=384, sft_batch_size=2, sft_grad_accum=16, sft_lr=5e-4,   grpo_lr=3e-5),
}
cfg = PRESETS[PRESET]
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Selected preset:', PRESET, cfg)

In [ ]:
# 3. 自前トークナイザー & 3段階カリキュラム生成器（疑似検索環境つき）
import copy, json, math, random, re, time
from contextlib import nullcontext
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from titans_pytorch import MemoryAsContextTransformer

torch.manual_seed(SEED)
random.seed(SEED)

SPECIAL_TOKENS = [
    '<|pad|>', '<|eos|>', '<|unk|>',
    '<think>', '</think>', '<search>', '</search>',
    '<result>', '</result>', '<answer>', '</answer>',
]

CONCEPT_NAMES = [
    'アルファ', 'ベータ', 'ガンマ', 'デルタ', 'シグマ', 'オメガ',
    'カッパ', 'ラムダ', 'ゼータ', 'イータ', 'ミュー', 'ニュー',
    'パイ', 'ロー', 'タウ', 'ファイ', 'カイ', 'プサイ',
    'ノードA', 'ノードB', 'ノードC', 'ノードD', 'ノードE', 'ノードF',
    '係数1', '係数2', '係数3', '係数4', '係数5', '係数6',
    '定数A', '定数B', '定数C', '定数D', '変数P', '変数Q',
]

DISTRACTOR_TURNS = [
    'メモ: 途中経過を保持中。',
    'メモ: 次の確認事項を待機中。',
    'メモ: 前の定義を記憶しておく。',
    'メモ: 文脈を継続して整理する。',
]

class CurriculumTokenizer:
    def __init__(self, token_to_id):
        self.token_to_id = dict(token_to_id)
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.pad_token_id = self.token_to_id['<|pad|>']
        self.eos_token_id = self.token_to_id['<|eos|>']
        self.unk_token_id = self.token_to_id['<|unk|>']
        self.search_end_id = self.token_to_id['</search>']
        self.vocab_size = len(self.token_to_id)
        self._split_re = re.compile('(' + '|'.join(re.escape(t) for t in SPECIAL_TOKENS) + ')')

    @classmethod
    def build_default(cls):
        chars = set('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz+-*=():.,!? \n')
        for s in [
            '質問: のとき、はいくつ？',
            '目的: を求める。既知: 不明な概念: 整理: にはが必要。記憶: よって',
            'エラー: 該当なし', '前回', *CONCEPT_NAMES, *DISTRACTOR_TURNS,
        ]:
            chars.update(s)
        vocab = {}
        for tok in SPECIAL_TOKENS:
            vocab[tok] = len(vocab)
        for ch in sorted(chars):
            if ch not in vocab:
                vocab[ch] = len(vocab)
        return cls(vocab)

    def encode(self, text):
        ids = []
        for part in self._split_re.split(text):
            if not part:
                continue
            if part in self.token_to_id:
                ids.append(self.token_to_id[part])
            else:
                for ch in part:
                    ids.append(self.token_to_id.get(ch, self.unk_token_id))
        return ids

    def decode(self, ids, skip_special_pad=True):
        out = []
        for i in ids:
            tok = self.id_to_token.get(i, '')
            if skip_special_pad and tok == '<|pad|>':
                continue
            out.append(tok)
        return ''.join(out)

@dataclass
class Episode:
    level: int
    prompt: str
    kb: dict
    expected_answer: int
    ideal_completion: str
    memory_reuse_target: bool = False

def _apply_op(val, op, k):
    return val + k if op == '+' else val * k

def generate_episode(level, rng):
    if level == 1:
        c1 = rng.choice(CONCEPT_NAMES)
        x = rng.randint(2, 15)
        op = rng.choice(['+', '*'])
        k = rng.randint(2, 9) if op == '+' else rng.randint(2, 5)
        ans = _apply_op(x, op, k)
        rule = f'{c1} = X {op} {k}'
        prompt = f'質問: X={x} のとき、{c1} はいくつ？\n'
        ideal = (
            f'<think>目的: {c1}を求める。既知: X={x}。不明な概念: {c1}</think>'
            f'<search>{c1}</search><result>{rule}</result>'
            f'<think>整理: {x} {op} {k} = {ans}</think><answer>{ans}</answer><|eos|>'
        )
        return Episode(1, prompt, {c1: rule}, ans, ideal)

    if level == 2:
        c1, c2 = rng.sample(CONCEPT_NAMES, 2)
        x = rng.randint(2, 10)
        op1 = rng.choice(['+', '*'])
        k1 = rng.randint(2, 8) if op1 == '+' else rng.randint(2, 4)
        v1 = _apply_op(x, op1, k1)
        op2 = rng.choice(['+', '*'])
        k2 = rng.randint(2, 8) if op2 == '+' else rng.randint(2, 4)
        ans = _apply_op(v1, op2, k2)
        r1, r2 = f'{c1} = X {op1} {k1}', f'{c2} = {c1} {op2} {k2}'
        prompt = f'質問: X={x} のとき、{c2} はいくつ？\n'
        ideal = (
            f'<think>目的: {c2}を求める。既知: X={x}。不明な概念: {c2}</think>'
            f'<search>{c2}</search><result>{r2}</result>'
            f'<think>整理: {c2}には{c1}が必要。不明な概念: {c1}</think>'
            f'<search>{c1}</search><result>{r1}</result>'
            f'<think>整理: {c1} = {x} {op1} {k1} = {v1}。{c2} = {v1} {op2} {k2} = {ans}</think>'
            f'<answer>{ans}</answer><|eos|>'
        )
        return Episode(2, prompt, {c1: r1, c2: r2}, ans, ideal)

    c1 = rng.choice(CONCEPT_NAMES)
    x1 = rng.randint(2, 12)
    x2 = rng.randint(2, 12)
    while x2 == x1:
        x2 = rng.randint(2, 12)
    op = rng.choice(['+', '*'])
    k = rng.randint(2, 9) if op == '+' else rng.randint(2, 5)
    ans1, ans2 = _apply_op(x1, op, k), _apply_op(x2, op, k)
    rule = f'{c1} = X {op} {k}'
    turn1 = (
        f'質問: X={x1} のとき、{c1} はいくつ？\n'
        f'<think>目的: {c1}を求める。既知: X={x1}。不明な概念: {c1}</think>'
        f'<search>{c1}</search><result>{rule}</result>'
        f'<think>整理: {x1} {op} {k} = {ans1}</think><answer>{ans1}</answer>\n'
        f'{rng.choice(DISTRACTOR_TURNS)}\n'
    )
    prompt = f'{turn1}質問: X={x2} のとき、{c1} はいくつ？\n'
    ideal = f'<think>目的: {c1}を求める。既知: X={x2}。記憶: {rule}。整理: {x2} {op} {k} = {ans2}</think><answer>{ans2}</answer><|eos|>'
    return Episode(3, prompt, {c1: rule}, ans2, ideal, memory_reuse_target=True)

tokenizer = CurriculumTokenizer.build_default()
(OUTPUT_DIR / 'vocab.json').write_text(json.dumps(tokenizer.token_to_id, ensure_ascii=False, indent=2), encoding='utf-8')
print('Vocab size:', tokenizer.vocab_size)

# サンプル表示
demo_rng = random.Random(0)
for lvl in (1, 2, 3):
    ep = generate_episode(lvl, demo_rng)
    print(f'\n--- Level {lvl} サンプル (tokens={len(tokenizer.encode(ep.prompt + ep.ideal_completion))}) ---')
    print(ep.prompt + ep.ideal_completion)

In [ ]:
# 4. Titans (MAC) モデル構築 & 疑似検索ロールアウト・評価関数
if torch.cuda.is_available():
    device = torch.device('cuda')
    major, _ = torch.cuda.get_device_capability()
    amp_dtype = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else (torch.float16 if major >= 7 else torch.float32)
else:
    device, amp_dtype = torch.device('cpu'), torch.float32

use_amp = device.type == 'cuda' and amp_dtype in (torch.bfloat16, torch.float16)
use_scaler = device.type == 'cuda' and amp_dtype == torch.float16

model = MemoryAsContextTransformer(
    num_tokens=tokenizer.vocab_size,
    dim=cfg['dim'],
    depth=cfg['depth'],
    segment_len=cfg['segment_len'],
    num_persist_mem_tokens=cfg['num_persist_mem_tokens'],
    num_longterm_mem_tokens=cfg['num_longterm_mem_tokens'],
).to(device)
print(f'Titans (MAC) params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | device={device} | amp={amp_dtype}')

SEARCH_RE = re.compile(r'<search>([^<>]+)</search>')
ANSWER_RE = re.compile(r'<answer>\s*(-?\d+)\s*</answer>')
THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def score_rollout(ep, comp_text, searches):
    format_ok = bool(THINK_RE.search(comp_text)) and (ANSWER_RE.search(comp_text) is not None)
    reward = 0.2 if format_ok else 0.0
    correct = False
    m = ANSWER_RE.search(comp_text)
    if m:
        try:
            if int(m.group(1)) == ep.expected_answer:
                correct = True
                reward += 1.0
        except ValueError:
            pass
    invalid = sum(1 for q in searches if q not in ep.kb)
    if invalid > 0:
        reward -= 0.1 * invalid
    if len(searches) > 3:
        reward -= 0.2
    if ep.memory_reuse_target and correct and len(searches) == 0:
        reward += 0.3
    return reward, correct, format_ok

@torch.no_grad()
def run_interactive_rollout(model, ep, max_new_tokens=MAX_NEW_TOKENS, temperature=0.8, top_p=0.9, greedy=False):
    model.eval()
    prompt_ids = tokenizer.encode(ep.prompt)
    seq = list(prompt_ids)
    action_mask = [0] * len(prompt_ids)
    searches = []
    for _ in range(max_new_tokens):
        if len(seq) >= cfg['seq_len']:
            break
        x = torch.tensor([seq], dtype=torch.long, device=device)
        logits = model(x)[:, -1, :]
        if greedy or temperature <= 1e-5:
            next_id = int(torch.argmax(logits, dim=-1).item())
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            remove = torch.cumsum(sorted_probs, dim=-1) > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = False
            sorted_probs[remove] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            next_id = int(sorted_idx.gather(-1, torch.multinomial(sorted_probs, 1)).item())
        seq.append(next_id)
        action_mask.append(1)
        if next_id == tokenizer.eos_token_id:
            break
        if next_id == tokenizer.search_end_id and len(searches) < 3:
            matches = SEARCH_RE.findall(tokenizer.decode(seq[len(prompt_ids):]))
            if matches:
                q = matches[-1].strip()
                searches.append(q)
                res = ep.kb.get(q, 'エラー: 該当なし')
                env_ids = tokenizer.encode(f'<result>{res}</result>')
                if len(seq) + len(env_ids) < cfg['seq_len']:
                    seq.extend(env_ids)
                    action_mask.extend([0] * len(env_ids))
    comp_text = tokenizer.decode(seq[len(prompt_ids):])
    reward, correct, format_ok = score_rollout(ep, comp_text, searches)
    return dict(full_ids=seq, action_mask=action_mask, comp_text=comp_text, searches=searches, reward=reward, correct=correct, format_ok=format_ok)

def evaluate_all(model, n_per_level=EVAL_PER_LEVEL, seed=9999):
    rng = random.Random(seed)
    results = {}
    for lvl in (1, 2, 3):
        c_cnt = f_cnt = m_cnt = 0
        r_sum = 0.0
        shown = False
        for _ in range(n_per_level):
            ep = generate_episode(lvl, rng)
            ro = run_interactive_rollout(model, ep, greedy=True)
            c_cnt += int(ro['correct'])
            f_cnt += int(ro['format_ok'])
            r_sum += ro['reward']
            if lvl == 3 and ro['correct'] and len(ro['searches']) == 0:
                m_cnt += 1
            if not shown:
                shown = True
                print(f'  [L{lvl} 例] 正解={ep.expected_answer} | 判定={ro["correct"]} | 検索={ro["searches"]}')
                print(f'         出力: {ro["comp_text"][:220]}')
        results[f'level_{lvl}'] = {
            'accuracy': round(c_cnt / n_per_level, 3),
            'format_rate': round(f_cnt / n_per_level, 3),
            'avg_reward': round(r_sum / n_per_level, 3),
            **({'memory_reuse_rate': round(m_cnt / n_per_level, 3)} if lvl == 3 else {}),
        }
    return results

In [ ]:
# 5. Stage 1: カリキュラム SFT（基礎作り）
def tokenize_sft(ep):
    p_ids = tokenizer.encode(ep.prompt)
    c_ids = tokenizer.encode(ep.ideal_completion)
    ids = p_ids + c_ids
    lbls = [-100] * len(p_ids) + list(c_ids)
    res_s, res_e = tokenizer.token_to_id['<result>'], tokenizer.token_to_id['</result>']
    in_res = False
    for i in range(len(p_ids), len(lbls)):
        if ids[i] == res_s:
            in_res = True
            lbls[i] = -100
        elif in_res:
            lbls[i] = -100
            if ids[i] == res_e:
                in_res = False
    return ids[:cfg['seq_len']], lbls[:cfg['seq_len']]

def sample_level(progress):
    r = random.random()
    if progress < 0.35:
        return 1 if r < 0.80 else 2
    if progress < 0.70:
        return 1 if r < 0.30 else (2 if r < 0.85 else 3)
    return 1 if r < 0.20 else (2 if r < 0.55 else 3)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['sft_lr'], betas=(0.9, 0.95), weight_decay=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=use_scaler)
sft_rng = random.Random(SEED)
warmup = max(1, min(80, SFT_STEPS // 10))

model.train()
t0 = time.time()
for step in range(1, SFT_STEPS + 1):
    progress = (step - 1) / max(1, SFT_STEPS)
    if step <= warmup:
        lr = cfg['sft_lr'] * step / warmup
    else:
        p = (step - warmup) / max(1, SFT_STEPS - warmup)
        lr = cfg['sft_lr'] * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * p)))
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for _ in range(cfg['sft_grad_accum']):
        batch = [tokenize_sft(generate_episode(sample_level(progress), sft_rng)) for _ in range(cfg['sft_batch_size'])]
        mlen = max(len(a) for a, _ in batch)
        x = torch.full((len(batch), mlen), tokenizer.pad_token_id, dtype=torch.long)
        y = torch.full((len(batch), mlen), -100, dtype=torch.long)
        for i, (a, b) in enumerate(batch):
            x[i, :len(a)] = torch.tensor(a)
            y[i, :len(b)] = torch.tensor(b)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with (torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()):
            logits = model(x[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1), ignore_index=-100)
            scaled = loss / cfg['sft_grad_accum']
        if use_scaler:
            scaler.scale(scaled).backward()
        else:
            scaled.backward()
        accum_loss += float(loss.item()) / cfg['sft_grad_accum']

    if use_scaler:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    if step % 50 == 0 or step == 1 or step == SFT_STEPS:
        print(f'[SFT] step {step:4d}/{SFT_STEPS} | loss {accum_loss:.4f} | lr {lr:.2e} | {time.time() - t0:.1f}s')

torch.save({'stage': 'sft', 'model_state_dict': model.state_dict()}, OUTPUT_DIR / 'stage1_sft_latest.pt')
if device.type == 'cuda':
    torch.cuda.empty_cache()

print('\n=== Stage 1 (SFT) 完了後の評価 ===')
sft_metrics = evaluate_all(model)
print(json.dumps(sft_metrics, ensure_ascii=False, indent=2))

In [ ]:
# 6. Stage 2: GRPO（誤り前提の強化学習・省メモリ実装）
def seq_logprobs(m, full_ids, action_mask):
    x = torch.tensor([full_ids[:-1]], dtype=torch.long, device=device)
    tgt = torch.tensor([full_ids[1:]], dtype=torch.long, device=device)
    mask = torch.tensor([action_mask[1:]], dtype=torch.float32, device=device)
    with (torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()):
        lp = F.log_softmax(m(x).float(), dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return lp.squeeze(0), mask.squeeze(0)

ref_model = copy.deepcopy(model).to(device).eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

rl_opt = torch.optim.AdamW(model.parameters(), lr=cfg['grpo_lr'], betas=(0.9, 0.95), weight_decay=0.01)
rl_rng = random.Random(SEED + 1000)
t0 = time.time()
ema_r = ema_acc = 0.0

for step in range(1, GRPO_STEPS + 1):
    progress = (step - 1) / max(1, GRPO_STEPS)
    r_lvl = rl_rng.random()
    lvl = (1 if r_lvl < 0.45 else (2 if r_lvl < 0.85 else 3)) if progress < 0.30 else (1 if r_lvl < 0.15 else (2 if r_lvl < 0.55 else 3))
    ep = generate_episode(lvl, rl_rng)

    rollouts = [run_interactive_rollout(model, ep, greedy=False) for _ in range(GROUP_SIZE)]
    rewards = [ro['reward'] for ro in rollouts]
    mean_r = sum(rewards) / GROUP_SIZE
    std_r = math.sqrt(sum((r - mean_r) ** 2 for r in rewards) / GROUP_SIZE)
    acc_g = sum(1.0 for ro in rollouts if ro['correct']) / GROUP_SIZE
    ema_r = mean_r if step == 1 else 0.9 * ema_r + 0.1 * mean_r
    ema_acc = acc_g if step == 1 else 0.9 * ema_acc + 0.1 * acc_g

    if std_r < 1e-6:
        if step % 10 == 0 or step == 1 or step == GRPO_STEPS:
            print(f'[GRPO] step {step:3d}/{GRPO_STEPS} | L{lvl} | r={mean_r:.3f} (ema {ema_r:.3f}) | acc={acc_g:.2f} | skip(zero-var)')
        continue

    advs = [(r - mean_r) / (std_r + 1e-6) for r in rewards]
    model.train()
    rl_opt.zero_grad(set_to_none=True)
    total_kl = 0.0
    for ro, adv in zip(rollouts, advs):
        if sum(ro['action_mask'][1:]) == 0:
            continue
        pol_lp, mask = seq_logprobs(model, ro['full_ids'], ro['action_mask'])
        with torch.no_grad():
            ref_lp, _ = seq_logprobs(ref_model, ro['full_ids'], ro['action_mask'])
        denom = mask.sum().clamp_min(1.0)
        log_ratio = (ref_lp - pol_lp).clamp(-10.0, 10.0)
        kl = ((torch.exp(log_ratio) - log_ratio - 1.0) * mask).sum() / denom
        pg = -adv * (pol_lp * mask).sum() / denom
        ((pg + KL_BETA * kl) / GROUP_SIZE).backward()
        total_kl += float(kl.item()) / GROUP_SIZE

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    rl_opt.step()
    if device.type == 'cuda' and step % 10 == 0:
        torch.cuda.empty_cache()
    if step % 10 == 0 or step == 1 or step == GRPO_STEPS:
        print(f'[GRPO] step {step:3d}/{GRPO_STEPS} | L{lvl} | r={mean_r:.3f} (ema {ema_r:.3f}) | acc={acc_g:.2f} (ema {ema_acc:.2f}) | kl={total_kl:.4f} | {time.time() - t0:.1f}s')

torch.save({'stage': 'grpo', 'model_state_dict': model.state_dict()}, OUTPUT_DIR / 'stage2_grpo_latest.pt')
del ref_model
if device.type == 'cuda':
    torch.cuda.empty_cache()

print('\n=== Stage 2 (GRPO) 完了後の評価 ===')
grpo_metrics = evaluate_all(model)
print(json.dumps(grpo_metrics, ensure_ascii=False, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps({'after_sft': sft_metrics, 'after_grpo': grpo_metrics}, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
# 7. 自由テストセル（任意の概念ルールで検索→論理整理の動きを確認）
custom_ep = Episode(
    level=2,
    prompt='質問: X=6 のとき、シグマ はいくつ？\n',
    kb={'シグマ': 'シグマ = アルファ * 3', 'アルファ': 'アルファ = X + 4'},
    expected_answer=30,
    ideal_completion='',
)
ro = run_interactive_rollout(model, custom_ep, greedy=True)
print('質問:', custom_ep.prompt.strip())
print('検索履歴:', ro['searches'])
print('正解判定:', ro['correct'], '| 報酬:', ro['reward'])
print('思考＋回答:\n', ro['comp_text'])